In [ ]:
#This version is a full variable load version

In [1]:
 
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
netCDF=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
true_time=netCDF['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
times=netCDF['time'].values/(1e9 * 60); times=times.astype(float);

#Restricts the timesteps of the data from timesteps0 to 140
data=netCDF.isel(time=np.arange(0,140+1))
parcel=parcel.isel(time=np.arange(0,140+1))

# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# parcel=parcel.isel(time=np.arange(0,400+1))

In [ ]:
###########################################################################################################################################################################

In [ ]:
#LOADING VARIABLES
###############################################################

In [2]:
# Loading Important Variables
##############
if 'emptylike' not in globals():
    print('loading neccessary variables')
    variable='lfc'; LFC_data=data[variable].data #get w data
    variable='lcl'; LCL_data=data[variable].data #get w data
    parcel_z=parcel['z'].data
    print('done')
    empty_like=True 

loading neccessary variables
done


In [1]:
def call_variables(t):
    variable='lfc'; LFC_data=data[variable].isel(time=t).data #get w data
    variable='lcl'; LCL_data=data[variable].isel(time=t).data #get w data
    parcel_z=parcel['z'].isel(time=t).data

    return LFC_data,LCL_data,parcel_z

In [3]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
with h5py.File(dir2+'lagrangian_binary_array.h5', 'r') as f:
    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)

In [ ]:
#MAKING LAGRANGIAN BINARY ARRAY
###############################################################

In [17]:
LFC=np.zeros_like(Z,dtype='float32')
LCL=np.zeros_like(Z,dtype='float32')

Nt=len(data['time'])
Np=len(parcel['xh'])
for p in np.arange(Np):
    if np.mod(p,25e3)==0: print(f"{p}/{len(parcel['xh'])}")

    #Get Indicies
    zs=parcel_z[:,p]
    ys=Y[:,p]
    xs=X[:,p]
    ts = np.arange(Nt)  

    
    #Get Values
    for t in np.arange(Nt):
        [LFC_data,LCL_data,parcel_z]=call_variables(t)
            
        lfcs = LFC_data[ys[t], xs[t]]
        lcls = LCL_data[ys[t], xs[t]]
        
        LFC[t,p]=(zs>=lfcs)*1
        LCL[t,p]=(zs>=lcls)*1

0/125000
25000/125000
50000/125000
75000/125000
100000/125000


In [19]:
# Saving Data
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
with h5py.File(dir2+f'LFC_LCL_binary_array.h5', 'w') as f:
    # Save the array as a variable in the file
    f.create_dataset('LFC', data=LFC) #binary array for general updraft (w>=0.1)
    f.create_dataset('LCL', data=LCL) #binary array for general updraft (w>=0.5 & qc+qi>=1e-6)

In [ ]:
#########################################

In [21]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
with h5py.File(dir2+f'LFC_LCL_binary_array.h5', 'r') as f:
    # Load the dataset by its name
    LFC = f['LFC'][:]
    LCL = f['LCL'][:]